# Demo: classify products (Step 1)

Expected output: `demo/products_categorized_output.csv`

Provided reference file: `demo/Products_dat/products_categories.csv`


In [1]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')

os.chdir('..')
!ls

README.md       demo            product_impacts


In [2]:
import numpy as np
import pandas as pd
import joblib

from sentence_transformers import SentenceTransformer
from product_impacts.ml_pipeline.make_predictions import get_predictions

/opt/homebrew/Caskroom/miniforge/base/envs/prod_imp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Configuration
data_dir = 'demo/'
products_file = 'products.csv'
models_dir = 'demo/trained_models/'
ndns_file = 'demo/Data Inputs/ndns_edited.csv'
model_type = 'nn'

# Categories that don't have a level-2 (sub food group) model 
no_lev2_categories = ['Savoury Snacks', 'Alcoholic Beverages', 'Nuts and Seeds', 'Not Food',
                      'Dietary Supplements', 'Artificial Sweeteners', 'Commercial Toddlers Foods and Drinks']

In [4]:
products = pd.read_csv(f'{data_dir}{products_file}')
products = products.fillna('')
products.shape

(500, 5)

## generate embeddings

Encodes each product's name + ingredients text with a sentence-transformer model. This writes two files into `demo/embeddings/`:

- `embeddings_0.npy` - the embedding vectors 
- `ids_0.npy` - the `product_id` for each row 

In [5]:
products['text'] = products['product_name'].str.lower() + ' - ' + products['ingredients_text'].str.lower()

model = SentenceTransformer('all-mpnet-base-v2')
query_embeddings = model.encode(products['text'].values.tolist())

os.makedirs(f'{data_dir}embeddings/', exist_ok=True)
np.save(f'{data_dir}embeddings/embeddings_0.npy', query_embeddings)
np.save(f'{data_dir}embeddings/ids_0.npy', np.array(products['product_id'].values))
print('embeddings done:', query_embeddings.shape)

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 5956.08it/s]


embeddings done: (500, 768)


## classify products

Two-level classification using the pre-trained neural network models in `demo/trained_models/`:
first a level-0 model assigns each product a broad parent category (e.g. "Fruit"), then a level-2 model assigns a more specific sub food group.

In [6]:
def get_ndns_cats(ndns_file):
    ndns = pd.read_csv(ndns_file, encoding='unicode_escape')
    ndns.loc[ndns['subfoodgroupdesc_edited'].notnull(),
             'subfoodgroupdesc'] = ndns[ndns['subfoodgroupdesc_edited'].notnull()]['subfoodgroupdesc_edited']
    ndns = ndns.drop(['subfoodgroupdesc_edited'], axis=1)
    ndns = ndns[['parentcategory', 'mainfoodgroupdesc', 'subfoodgroupdesc']].drop_duplicates()
    ndns = ndns[ndns['subfoodgroupdesc'] != 'Tap water only']
    ndns = ndns.rename(columns={'mainfoodgroupdesc': 'mainfoodgroup', 'subfoodgroupdesc': 'subfoodgroup'})
    return ndns

ndns = get_ndns_cats(ndns_file)
ndns['parentcategory'].unique()

array(['Cereals and Cereal Products', 'Milk and Milk Products',
       'Eggs and Egg Dishes', 'Fat Spreads', 'Meat and Meat Products',
       'Fish and Fish Dishes', 'Vegetables, Potatoes', 'Savoury Snacks',
       'Nuts and Seeds', 'Fruit', 'Sugar, Preserves and Confectionery',
       'Non-Alcoholic Beverages', 'Alcoholic Beverages', 'Miscellaneous',
       'Commercial Toddlers Foods and Drinks', 'Dietary Supplements',
       'Artificial Sweeteners'], dtype=object)

In [7]:
# Loading embeddings 
pred_emb = np.load(f'{data_dir}embeddings/embeddings_0.npy')
pred_product_ids = np.load(f'{data_dir}embeddings/ids_0.npy')

pred_features = pd.DataFrame(data=pred_emb)
X_cols = pred_features.columns.tolist()
pred_features['product_id'] = pd.Series(pred_product_ids, index=pred_features.index)

pred = products.merge(pred_features)
pred.shape

(500, 774)

In [8]:
# Level 0: assign each product a parent category
stored_model = joblib.load(f'{models_dir}{model_type}_lev0.joblib')
X_pred = pred[X_cols].to_numpy()

y_pred, y_pred_probabilities = get_predictions(stored_model, X_pred, model_type)
pred['parentcategory'] = pd.Series(y_pred, index=pred.index)
pred['parentcategory_prob'] = pd.Series(y_pred_probabilities, index=pred.index)

pred['parentcategory'].value_counts()

 1/16 [>.............................] - ETA: 1s

/opt/homebrew/Caskroom/miniforge/base/envs/prod_imp/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.4.1.post1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


16/16 [==============================] - 0s 1ms/step


parentcategory
Cereals and Cereal Products           481
Milk and Milk Products                  6
Vegetables, Potatoes                    3
Fish and Fish Dishes                    2
Meat and Meat Products                  2
Savoury Snacks                          1
Nuts and Seeds                          1
Eggs and Egg Dishes                     1
Miscellaneous                           1
Fat Spreads                             1
Sugar, Preserves and Confectionery      1
Name: count, dtype: int64

In [9]:
# Level 2: for categories with a sub-category model, assign main/sub food group
df_list = []

for category in ndns['parentcategory'].unique().tolist():

    pred_df = pred[pred['parentcategory'] == category].copy()
    if pred_df.empty:
        continue

    print(category, pred_df.shape)

    if category in no_lev2_categories:
        pred_df['mainfoodgroup'] = pred_df['parentcategory']
        pred_df['subfoodgroup'] = pred_df['parentcategory']
    else:
        stored_model = joblib.load(f'{models_dir}{model_type}_{category}.joblib')
        X_pred = pred_df[X_cols].to_numpy()

        y_pred2, y_pred2_probabilities = get_predictions(stored_model, X_pred, model_type)

        y_pred1 = pd.DataFrame(y_pred2, columns=['subfoodgroup']).merge(
            ndns[['mainfoodgroup', 'subfoodgroup']].drop_duplicates(), how='left')['mainfoodgroup'].values
        pred_df['mainfoodgroup'] = pd.Series(y_pred1, index=pred_df.index)
        pred_df['subfoodgroup'] = pd.Series(y_pred2, index=pred_df.index)
        pred_df['subfoodgroup_prob'] = pd.Series(y_pred2_probabilities, index=pred_df.index)

    df_list.append(pred_df)

pred = pd.concat(df_list, axis=0, ignore_index=True)
pred.shape

Cereals and Cereal Products (481, 776)
16/16 [==============================] - 0s 786us/step
Milk and Milk Products (6, 776)


/opt/homebrew/Caskroom/miniforge/base/envs/prod_imp/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.4.1.post1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/prod_imp/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.4.1.post1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


1/1 [==============================] - 0s 39ms/step


Eggs and Egg Dishes (1, 776)
1/1 [==============================] - 0s 27ms/step


/opt/homebrew/Caskroom/miniforge/base/envs/prod_imp/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.4.1.post1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/prod_imp/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.4.1.post1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


...

/opt/homebrew/Caskroom/miniforge/base/envs/prod_imp/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.4.1.post1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


1/1 [==============================] - 0s 46ms/step
Miscellaneous (1, 776)


/opt/homebrew/Caskroom/miniforge/base/envs/prod_imp/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.4.1.post1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


1/1 [==============================] - 0s 41ms/step


(500, 779)

## Save output

Renaming columns to match the reference file 

In [10]:
output = pred[['product_id', 'product_name', 'ingredients_text', 'country', 'country_group',
               'parentcategory', 'mainfoodgroup', 'subfoodgroup']].rename(columns={
    'country_group': 'Retailer',
    'parentcategory': 'Department',
    'mainfoodgroup': 'Aisle',
    'subfoodgroup': 'Shelf',
})

output.to_csv(f'{data_dir}products_categorized_output.csv', index=False)
output.head()

,product_id,product_name,ingredients_text,country,Retailer,Department,Aisle,Shelf
0,291120242460529,Lye football,"wheat flour, water, yeast, rapeseed oil, dinin...",Austria,EUwest,Cereals and Cereal Products,Bread,"White bread (not high fibre, not multiseed bread)"
1,291120242558123,Helical,durum wheat semolina,France,EUwest,Cereals and Cereal Products,"Pasta, rice and other miscellaneous cereals",Other cereals
2,291120242035935,Natural bread,"t65 wheat flour, filtered water, natural sourd...",France,EUwest,Cereals and Cereal Products,Bread,"White bread (not high fibre, not multiseed bread)"
3,291120242009849,Rice Crackers Taste Chile,"rice powder, palm oil, sugar, soy sauce, seawe...",France,EUwest,Cereals and Cereal Products,Biscuits,Biscuits
4,291120242332342,Instant noodles soup beef flavour,"noodles (92%) (wheat flour , refined palm oil ...",France,EUwest,Cereals and Cereal Products,"Pasta, rice and other miscellaneous cereals",Pasta
